![Project](https://img.shields.io/badge/NS%20Health%20%26%20Population%20Analytics-1B3A6B?style=for-the-badge&logoColor=white)

# 💰 Income by Age & Gender — Data Cleaning Pipeline
![Python](https://img.shields.io/badge/Python_3.12-C9A020?style=flat-square&logo=python&logoColor=white) ![pandas](https://img.shields.io/badge/pandas-1A6B5A?style=flat-square&logo=pandas&logoColor=white) ![Source](https://img.shields.io/badge/Source-Statistics_Canada-1B3A6B?style=flat-square)

> **Analytical Question:** Is there a correlation between socioeconomic indicators (income, education) and health outcomes in NS communities?

This notebook cleans 12 StatCan income CSV files — split across gender (Males / Females) and age groups (15–24, 25–34, 35–44, 45–54, 55–64, 65+) — and stacks them into a single analysis-ready dataset.

---

## 📦 Source Data

| Files | Gender | Age Groups | Years | Source Table |
|-------|--------|------------|-------|-------------|
| `Men Income (1–6).csv` | Males | 25–34, 35–44, 45–54, 55–64, 65+ | 2012–2023 | StatCan 11-10-0239-01 |
| `Women (1–6).csv` | Females | 15–24, 25–34, 45–54, 55–64, 65+ | 2012–2023 | StatCan 11-10-0239-01 |

**Source:** Statistics Canada, Table 11-10-0239-01 — *Income of individuals by age group, sex and income source. 2023 constant dollars.*

---

## ⚠️ Known Data Quality Issues

| Issue | Detail | Handling |
|-------|--------|---------|
| Quality flags on values | Letters A–E appended to numbers (e.g. `54500C`) | Stripped; stored in `Data Quality` column |
| Suppressed values | `'x'` = confidentially suppressed | Replaced with `NaN` via `errors='coerce'` |
| No Male 15–24 file | Age band absent from upload | Noted in data dictionary; gap in analysis |

**StatCan Quality Flag Reference:**

| Flag | Meaning |
|------|---------|
| A | Excellent (CV < 2%) |
| B | Very Good (CV 2–5%) |
| C | Good (CV 5–10%) — use with caution |
| D | Acceptable (CV 10–15%) — flag in visuals |
| E | Use with caution (CV 15–25%) |
| x | Suppressed — treat as NULL |

---

## 🔧 Cleaning Steps Overview

| Step | Action |
|------|--------|
| 1 | Imports & paths |
| 2 | Clean one income file — rename, select, cast types |
| 3 | Discover and process all 12 files |
| 4 | Combine, sort, export |

---


## Step 1 — Imports & Paths


In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR  = Path("data")
CLEAN_DIR = Path("clean")
CLEAN_DIR.mkdir(exist_ok=True)

OUTPUT_CSV = CLEAN_DIR / "income_cleaned.csv"


## Step 2 — Clean One Income File

The StatCan income CSVs use a modern tidy structure (unlike the wide population files). Each row is already one observation. The main cleaning tasks are:

1. **Rename columns** — source uses `REF_DATE`, `GEO`, `STATUS` etc.; we standardize to `Year`, `Geography`, `Data Quality`
2. **Select only needed columns** — drop the many StatCan metadata columns we don't need
3. **Cast types** — `Year` to int, `Value` to numeric (`errors='coerce'` handles suppressed `'x'` values and quality letter flags)

> The `STATUS` column in StatCan files contains the quality flag (A/B/C/D/E/x). We rename it to `Data Quality` for clarity.


In [ ]:
def clean_income_file(path):
    """
    Cleans one StatCan income CSV.

    Key operations:
    - Rename StatCan column names to human-readable labels
    - Keep only the 8 columns needed for analysis
    - Cast Year to int, Value to float (errors='coerce' handles 'x' suppressions)
    """
    df = pd.read_csv(path, encoding="utf-8-sig")

    df = df.rename(columns={
        "REF_DATE"      : "Year",
        "GEO"           : "Geography",
        "Sex"           : "Gender",
        "Income source" : "Income source",
        "Statistics"    : "Statistic",
        "VALUE"         : "Value",
        "STATUS"        : "Data Quality"   # quality flag A/B/C/D/E/x
    })

    # Keep only the columns needed for analysis
    df = df[[
        "Year", "Geography", "Age group", "Gender",
        "Income source", "Statistic", "Value", "Data Quality"
    ]]

    # Cast types
    df["Year"]  = df["Year"].astype(int)
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    # errors='coerce' silently converts:
    #   'x'   (suppressed)   → NaN  (NULL in SQL terms)
    #   '54500C' (with flag) → NaN  (flag already in Data Quality column)

    return df


## Step 3 — Discover & Process All 12 Files

We use `pathlib.Path.glob()` to automatically discover all Men Income and Women files in the data folder, rather than hardcoding filenames. This makes the pipeline resilient to file renaming.


In [ ]:
income_files = (
    list(DATA_DIR.glob("Men Income*.csv")) +
    list(DATA_DIR.glob("Women*.csv"))
)

print(f"Found {len(income_files)} income files")

frames = []
for file in income_files:
    print(f"  Processing {file.name}")
    frames.append(clean_income_file(file))


## Step 4 — Combine, Sort & Export


In [ ]:
Income = (
    pd.concat(frames, ignore_index=True)
      .dropna(subset=["Value"])           # remove suppressed 'x' rows
      .sort_values(["Gender", "Age group", "Statistic", "Year"])
      .reset_index(drop=True)
)

Income.to_csv(OUTPUT_CSV, index=False)

print("✅ Income data cleaned successfully")
print(f"✅ Rows saved : {Income.shape[0]}")
print(f"✅ Output     : {OUTPUT_CSV}")
print()
print("── Data Quality Flag Distribution ──────────────")
print(Income["Data Quality"].value_counts().to_string())
print()
print(Income.head())
Income


## ✅ Output Summary

| Output | Location | Rows | Columns |
|--------|----------|------|---------|
| Clean CSV | `./clean/income_cleaned.csv` | ~720 | 8 |

**Final schema:** `Year` \| `Geography` \| `Age group` \| `Gender` \| `Income source` \| `Statistic` \| `Value` \| `Data Quality`

**Feeds into:**
- 📊 Power BI — scatter plot X-axis (avg income vs. Health Outcome Index)
- 🧮 `Health_Outcome_Index.ipynb` — income component of HOI composite score

> ⚠️ **Filter note:** For NS-specific analysis, filter `Geography = 'Nova Scotia'`. Other geographies (Canada, other provinces) are present for benchmark comparison.


---

*Nova Scotia Health & Population Analytics · Statistics Canada 11-10-0239-01*
